In [1]:
from os.path import join as pjoin
import pandas as pd
import numpy as np

from caveclient import CAVEclient

# What version to materialize from CAVE (latest)
version = 1718

# Initialize caveclient and set version
client = CAVEclient('minnie65_public')

client.version = version

In [2]:
# get cell info for the requested version
cell_df = client.materialize.views.aibs_cell_info().query(
    select_columns =['id','pt_root_id','broad_type','cell_type','mtype','meso_type','visual_area','pt_position'],
    materialization_version=version,
    split_positions=True,
)

if np.isin('pt_supervoxel_id', cell_df.columns):
    cell_df = cell_df.drop(columns = 'pt_supervoxel_id')

# Drop duplicates on pt_root_id, and pt_root_id=0
cell_df = cell_df.drop_duplicates(subset='pt_root_id', keep=False)

# Clean the manual cell categories for clarity
ct_mapping = {'WM-P': '6P-CT', '6P-U': '6P-IT', 'error': np.nan}
cell_df['cell_type'] = cell_df.cell_type.replace(ct_mapping)

# Add the functional data
functional_ids = client.materialize.tables.coregistration_manual_v4().query(
    materialization_version=version).id.values

cell_df['coreg'] = cell_df.id.isin(functional_ids)

# Add if cell is in the V1 Column
column_roots = client.materialize.tables.allen_v1_column_types_slanted_ref().query()['pt_root_id'].to_numpy()
cell_df['is_column'] = cell_df.pt_root_id.isin(column_roots)

Table Owner Notice on aibs_cell_info: Warning: this view is subject to change.


In [3]:
# Add the proofreading status
prf_df = client.materialize.tables.proofreading_status_and_strategy().query(
    select_columns = ['pt_root_id','valid_id','status_dendrite','status_axon',
                      'strategy_dendrite','strategy_axon'],
    materialization_version = version
)

if np.isin('pt_supervoxel_id', prf_df.columns):
        prf_df = prf_df.drop(columns = 'pt_supervoxel_id')

# Select only proofread where the root_id==valid_id (no further proofreading after validation)
prf_df = prf_df.query("pt_root_id==valid_id")
prf_df = prf_df.drop(columns=['valid_id'])

# Merge to cell types
cell_df = (cell_df.merge(prf_df, on='pt_root_id', how='left')
           .fillna({'status_dendrite': False,
                    'status_axon': False,
                    'strategy_dendrite': 'none',
                    'strategy_axon': 'none',
                   })
           .astype({'status_dendrite': bool, 'status_axon': bool})
          )

In [4]:
from standard_transform import minnie_ds

X_transformed = minnie_ds.transform_vx.apply_dataframe('pt_position', cell_df)
X_transformed = np.array(X_transformed)
cell_df['pt_position_x_tform'] = X_transformed[:,0]
cell_df['pt_position_y_tform'] = X_transformed[:,1]
cell_df['pt_position_z_tform'] = X_transformed[:,2] 

In [5]:
cell_df.query('status_axon').head(3)

,id,pt_root_id,broad_type,cell_type,mtype,meso_type,visual_area,pt_position_x,pt_position_y,pt_position_z,coreg,is_column,status_dendrite,status_axon,strategy_dendrite,strategy_axon,pt_position_x_tform,pt_position_y_tform,pt_position_z_tform
6442,106221,864691136619199963,inhibitory,BC,STC,AltBasket,V1,101712,120464,15401,False,False,True,True,dendrite_clean,axon_partially_extended,363.303303,118.810500,616.04
6839,107130,864691136275890189,inhibitory,BC,PTC,AltBasket,V1,111120,115984,23104,False,False,True,True,dendrite_clean,axon_partially_extended,402.353933,104.238536,924.16
12895,153534,864691135886221168,excitatory,23P,L2c,L2IT,V1,129312,118096,22292,True,False,True,True,dendrite_extended,axon_fully_extended,474.108737,118.996538,891.68


In [6]:
cell_df.to_csv('../docs/resources/data/v1718_cell_info.csv', index=False)